## 1. ETL PostgreSQL - MinIO

In [1]:
import os
import json
from pathlib import Path

import boto3
import joblib
import numpy as np
import pandas as pd
from sqlalchemy import create_engine, text
from sklearn.ensemble import GradientBoostingRegressor, IsolationForest
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler


def read_env(name, fallback):
    return os.getenv(name, fallback)


PG_SETTINGS = {
    'user': read_env('POSTGRES_USER', 'admin'),
    'password': read_env('POSTGRES_PASSWORD', 'admin'),
    'host': read_env('POSTGRES_HOST', 'postgres'),
    'port': read_env('POSTGRES_PORT', '5432'),
    'database': read_env('POSTGRES_DB', 'oil_analytics'),
}
MINIO_SETTINGS = {
    'endpoint': read_env('MINIO_ENDPOINT', 'http://minio:9000'),
    'bucket': read_env('MINIO_BUCKET', 'oil-lake'),
    'access_key': read_env('MINIO_ACCESS_KEY', 'admin'),
    'secret_key': read_env('MINIO_SECRET_KEY', 'adminadmin'),
}

POSTGRES_URL = (
    f"postgresql+psycopg2://{PG_SETTINGS['user']}:{PG_SETTINGS['password']}"
    f"@{PG_SETTINGS['host']}:{PG_SETTINGS['port']}/{PG_SETTINGS['database']}"
)
MINIO_ENDPOINT = MINIO_SETTINGS['endpoint']
MINIO_BUCKET = MINIO_SETTINGS['bucket']
S3_OPTIONS = {
    'key': MINIO_SETTINGS['access_key'],
    'secret': MINIO_SETTINGS['secret_key'],
    'client_kwargs': {'endpoint_url': MINIO_ENDPOINT},
}

engine = create_engine(POSTGRES_URL)
s3 = boto3.client(
    's3',
    endpoint_url=MINIO_ENDPOINT,
    aws_access_key_id=MINIO_SETTINGS['access_key'],
    aws_secret_access_key=MINIO_SETTINGS['secret_key'],
)

cwd = Path.cwd()
PROJECT_ROOT = cwd.parent if cwd.name in {'notebooks', 'scripts'} else cwd
MODELS_DIR = PROJECT_ROOT / 'models'
MODELS_DIR.mkdir(exist_ok=True)


In [2]:
EXPORT_PLAN = (
    {'table': 'wells'},
    {'table': 'production', 'date_column': 'date'},
    {'table': 'well_telemetry', 'date_column': 'timestamp'},
    {'table': 'well_targets', 'date_column': 'date'},
    {'table': 'pumps'},
    {'table': 'pump_sensors', 'date_column': 'timestamp'},
    {'table': 'pump_failures', 'date_column': 'failure_date'},
    {'table': 'deliveries', 'date_column': 'date'},
    {'table': 'drivers'},
    {'table': 'vehicles'},
    {'table': 'oil_stations'},
)

bucket_names = {bucket['Name'] for bucket in s3.list_buckets()['Buckets']}
if MINIO_BUCKET not in bucket_names:
    s3.create_bucket(Bucket=MINIO_BUCKET)


def coerce_dates(frame):
    prepared = frame.copy()
    candidates = [
        column for column in prepared.columns
        if column in {'date', 'timestamp', 'install_date', 'failure_date'} or column.endswith('_date')
    ]
    for column in candidates:
        prepared[column] = pd.to_datetime(prepared[column], errors='coerce')
    return prepared


def attach_partition(frame, date_column):
    prepared = coerce_dates(frame)
    prepared['dt'] = (
        prepared[date_column].dt.date
        if date_column else pd.Timestamp('2025-10-01').date()
    )
    return prepared


def export_source(job):
    table = job['table']
    raw = pd.read_sql(text(f'SELECT * FROM {table}'), engine)
    prepared = attach_partition(raw, job.get('date_column'))
    paths = {
        'parquet': f's3://{MINIO_BUCKET}/bronze/{table}',
        'csv': f's3://{MINIO_BUCKET}/csv/{table}.csv',
    }
    prepared.to_parquet(paths['parquet'], engine='pyarrow', partition_cols=['dt'], index=False, storage_options=S3_OPTIONS)
    prepared.to_csv(paths['csv'], index=False, storage_options=S3_OPTIONS)
    return {'table': table, 'rows': len(prepared), **paths}


export_report = [export_source(job) for job in EXPORT_PLAN]
objects = s3.list_objects_v2(Bucket=MINIO_BUCKET).get('Contents', [])

pd.DataFrame(export_report), pd.DataFrame(
    [{'key': item['Key'], 'size': item['Size']} for item in objects]
).head(30)


/opt/conda/lib/python3.12/site-packages/fsspec/registry.py:305: UserWarning: Your installed version of s3fs is very old and known to cause
severe performance issues, see also https://github.com/dask/dask/issues/10276

To fix, you should specify a lower version bound on s3fs, or
update the current installation.

  warnings.warn(s3_msg)


(             table  rows                              parquet  \
 0            wells     5           s3://oil-lake/bronze/wells   
 1       production   150      s3://oil-lake/bronze/production   
 2   well_telemetry    48  s3://oil-lake/bronze/well_telemetry   
 3     well_targets    90    s3://oil-lake/bronze/well_targets   
 4            pumps     5           s3://oil-lake/bronze/pumps   
 5     pump_sensors    72    s3://oil-lake/bronze/pump_sensors   
 6    pump_failures     3   s3://oil-lake/bronze/pump_failures   
 7       deliveries    30      s3://oil-lake/bronze/deliveries   
 8          drivers     5         s3://oil-lake/bronze/drivers   
 9         vehicles     5        s3://oil-lake/bronze/vehicles   
 10    oil_stations    20    s3://oil-lake/bronze/oil_stations   
 
                                      csv  
 0            s3://oil-lake/csv/wells.csv  
 1       s3://oil-lake/csv/production.csv  
 2   s3://oil-lake/csv/well_telemetry.csv  
 3     s3://oil-lake/csv/well_

## 2. Аналитические витрины

In [3]:
def iqr_clip(frame, columns):
    result = frame.copy()
    for col in columns:
        q1, q3 = result[col].quantile([0.25, 0.75])
        spread = q3 - q1
        if pd.notna(spread) and spread > 0:
            result[col] = result[col].clip(q1 - 1.5 * spread, q3 + 1.5 * spread)
    return result


def fill_by_group(frame, group_col, columns):
    order_col = 'date' if 'date' in frame.columns else 'timestamp'
    result = frame.sort_values([group_col, order_col]).copy()
    for col in columns:
        result[col] = result.groupby(group_col)[col].transform(lambda s: s.ffill().bfill())
        result[col] = result[col].fillna(result.groupby(group_col)[col].transform('median'))
        result[col] = result[col].fillna(result[col].median())
    return result

wells = pd.read_sql('SELECT * FROM wells', engine, parse_dates=['start_date'])
production = pd.read_sql('SELECT * FROM production', engine, parse_dates=['date'])
telemetry = pd.read_sql('SELECT * FROM well_telemetry', engine, parse_dates=['timestamp'])
deliveries = pd.read_sql('SELECT * FROM deliveries', engine, parse_dates=['date'])
drivers = pd.read_sql('SELECT * FROM drivers', engine)

prod_num = ['oil_ton', 'gas_m3', 'water_m3', 'energy_kwh', 'downtime_hours', 'temperature', 'pressure']
tel_num = ['pump_speed_rpm', 'pump_current', 'pressure_in', 'pressure_out', 'temperature', 'vibration', 'oil_flow_rate']

production = fill_by_group(production, 'well_id', prod_num)
production = iqr_clip(production, prod_num)
production['downtime_ratio'] = production['downtime_hours'] / 24

telemetry['date'] = telemetry['timestamp'].dt.date.astype('datetime64[ns]')
telemetry = fill_by_group(telemetry, 'well_id', tel_num)
telemetry = iqr_clip(telemetry, tel_num)
telemetry_daily = telemetry.groupby(['well_id', 'date'], as_index=False).agg(
    avg_pressure=('pressure_out', 'mean'),
    avg_temperature=('temperature', 'mean'),
    avg_current=('pump_current', 'mean'),
    pump_runtime_hours=('timestamp', 'count'),
    avg_vibration=('vibration', 'mean'),
)

fact = production.merge(telemetry_daily, on=['well_id', 'date'], how='left')
fact = fact.merge(wells[['well_id', 'name', 'field_name', 'region', 'status']], on='well_id', how='left')

daily_output_view = fact.groupby('date', as_index=False).agg(
    total_oil_ton=('oil_ton', 'sum'),
    avg_pressure=('avg_pressure', 'mean'),
    avg_temperature=('avg_temperature', 'mean'),
    downtime_ratio=('downtime_ratio', 'mean'),
)
well_scorecard_view = fact.groupby(['well_id', 'name', 'field_name', 'region'], as_index=False).agg(
    avg_flow_tpd=('oil_ton', 'mean'),
    total_oil_ton=('oil_ton', 'sum'),
    downtime_pct=('downtime_ratio', lambda s: s.mean() * 100),
    avg_pressure=('avg_pressure', 'mean'),
    avg_temperature=('avg_temperature', 'mean'),
)
pressure_flow_view = fact[['date', 'well_id', 'name', 'avg_pressure', 'avg_temperature', 'oil_ton', 'energy_kwh', 'downtime_ratio']].copy()

deliveries['cost_per_km'] = deliveries['cost_usd'] / deliveries['distance_km'].replace(0, np.nan)
deliveries['delay_rate'] = (deliveries['delay_hours'] > 0).astype(int)
deliveries['weather_severity'] = deliveries['weather_conditions'].map({'Clear': 0, 'Cloudy': 1, 'Fog': 2, 'Rain': 3, 'Snow': 4}).fillna(1)
logistics = deliveries.merge(drivers, on='driver_id', how='left')
delivery_scorecard_view = logistics.groupby(['driver_id', 'name', 'weather_conditions'], as_index=False).agg(
    avg_delay_hours=('delay_hours', 'mean'),
    delay_rate=('delay_rate', 'mean'),
    avg_cost_per_km=('cost_per_km', 'mean'),
    weather_severity=('weather_severity', 'mean'),
    total_volume_ton=('volume_ton', 'sum'),
    delivery_count=('delivery_id', 'count'),
    avg_distance_km=('distance_km', 'mean'),
)
delivery_scorecard_view['driver_reliability_score'] = 1 / (1 + delivery_scorecard_view['avg_delay_hours'] + delivery_scorecard_view['delay_rate'])

for name, frame in {
    'dm_daily_output': daily_output_view,
    'dm_well_scorecard': well_scorecard_view,
    'dm_pressure_flow': pressure_flow_view,
    'dm_delivery_scorecard': delivery_scorecard_view,
}.items():
    frame.to_sql(name, engine, if_exists='replace', index=False)

pd.read_sql('''
SELECT 'dm_daily_output' AS table_name, COUNT(*) AS rows_count FROM dm_daily_output
UNION ALL SELECT 'dm_well_scorecard', COUNT(*) FROM dm_well_scorecard
UNION ALL SELECT 'dm_pressure_flow', COUNT(*) FROM dm_pressure_flow
UNION ALL SELECT 'dm_delivery_scorecard', COUNT(*) FROM dm_delivery_scorecard
''', engine)

,table_name,rows_count
0,dm_daily_output,30
1,dm_well_scorecard,5
2,dm_pressure_flow,150
3,dm_delivery_scorecard,13


## 3. ML-прогноз дебита

In [4]:
targets = pd.read_sql('SELECT * FROM well_targets', engine, parse_dates=['date'])
production = pd.read_sql('SELECT * FROM production', engine, parse_dates=['date'])
telemetry = pd.read_sql('SELECT * FROM well_telemetry', engine, parse_dates=['timestamp'])
telemetry['date'] = telemetry['timestamp'].dt.date.astype('datetime64[ns]')

daily_telemetry = telemetry.groupby(['well_id', 'date'], as_index=False).agg(
    pressure=('pressure_out', 'mean'),
    temperature=('temperature', 'mean'),
    power_proxy=('pump_current', 'mean'),
    pump_runtime_hours=('timestamp', 'count'),
)
prod_features = production[['well_id', 'date', 'energy_kwh', 'downtime_hours']].copy()
ml_data = targets.merge(daily_telemetry, on=['well_id', 'date'], how='left').merge(prod_features, on=['well_id', 'date'], how='left')
feature_cols = ['pressure', 'temperature', 'power_proxy', 'pump_runtime_hours', 'energy_kwh', 'downtime_hours']
for col in feature_cols:
    ml_data[col] = ml_data.groupby('well_id')[col].transform(lambda s: s.ffill().bfill())
    ml_data[col] = ml_data[col].fillna(ml_data.groupby('well_id')[col].transform('median'))
    ml_data[col] = ml_data[col].fillna(ml_data[col].median())

X = ml_data[feature_cols]
y = ml_data['daily_oil_ton']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=42)
models = {'ridge': Ridge(alpha=1.0), 'gradient_boosting': GradientBoostingRegressor(random_state=42)}
scores = {}
for model_name, model in models.items():
    model.fit(X_train, y_train)
    predicted = model.predict(X_test)
    scores[model_name] = {
        'MAE': float(mean_absolute_error(y_test, predicted)),
        'RMSE': float(np.sqrt(mean_squared_error(y_test, predicted))),
        'MAPE': float(np.mean(np.abs((y_test - predicted) / y_test.replace(0, np.nan))) * 100),
        'R2': float(r2_score(y_test, predicted)),
    }

best_name = min(scores, key=lambda key: scores[key]['RMSE'])
best_model = models[best_name]
ml_data['predicted_flow_tpd'] = best_model.predict(X)
ml_data['absolute_error'] = (ml_data['daily_oil_ton'] - ml_data['predicted_flow_tpd']).abs()
ml_data.to_sql('dm_flow_predictions', engine, if_exists='replace', index=False)
metrics = {'best_model': best_name, 'test_size': 0.30, 'scores': scores}
joblib.dump(best_model, MODELS_DIR / 'flow_forecast_ridge_gb.joblib')
(MODELS_DIR / 'flow_forecast_metrics.json').write_text(json.dumps(metrics, indent=2), encoding='utf-8')
metrics

{'best_model': 'gradient_boosting',
 'test_size': 0.3,
 'scores': {'ridge': {'MAE': 0.5828351819242149,
   'RMSE': 0.779047082767371,
   'MAPE': 0.2868852061945461,
   'R2': 0.9952726890507853},
  'gradient_boosting': {'MAE': 0.2486150227612764,
   'RMSE': 0.3303507050379278,
   'MAPE': 0.12233918967644876,
   'R2': 0.9991499641650053}}}

## 4. Аномалии и риск отказа оборудования

In [5]:
sensors = pd.read_sql('SELECT * FROM pump_sensors', engine, parse_dates=['timestamp']).sort_values(['pump_id', 'timestamp'])
failures = pd.read_sql('SELECT * FROM pump_failures', engine, parse_dates=['failure_date'])
sensor_cols = ['vibration', 'temperature', 'current', 'rpm', 'pressure']
for col in sensor_cols:
    sensors[col] = sensors.groupby('pump_id')[col].transform(lambda s: s.ffill().bfill())
    sensors[col] = sensors[col].fillna(sensors.groupby('pump_id')[col].transform('median'))
    sensors[col] = sensors[col].fillna(sensors[col].median())

scaler = StandardScaler()
z_values = scaler.fit_transform(sensors[sensor_cols])
for idx, col in enumerate(sensor_cols):
    sensors[f'{col}_z'] = z_values[:, idx]
sensors['zscore_anomaly_flag'] = (np.abs(z_values) > 3).any(axis=1).astype(int)
sensors['isolation_anomaly'] = (IsolationForest(contamination=0.05, random_state=42).fit_predict(sensors[sensor_cols]) == -1).astype(int)
sensors['anomaly_flag'] = sensors[['zscore_anomaly_flag', 'isolation_anomaly']].max(axis=1)

sensors['failure_within_24h'] = 0
for failure in failures.itertuples(index=False):
    window = ((sensors['pump_id'] == failure.pump_id) & (sensors['timestamp'] <= failure.failure_date) & (sensors['timestamp'] >= failure.failure_date - pd.Timedelta(hours=24)))
    sensors.loc[window, 'failure_within_24h'] = 1

risk_features = sensor_cols + [f'{col}_z' for col in sensor_cols] + ['anomaly_flag']
if sensors['failure_within_24h'].nunique() > 1:
    train_x, test_x, train_y, test_y = train_test_split(sensors[risk_features], sensors['failure_within_24h'], test_size=0.30, random_state=42, stratify=sensors['failure_within_24h'])
    classifier = LogisticRegression(class_weight='balanced', max_iter=1000)
    classifier.fit(train_x, train_y)
    sensors['failure_risk_score'] = classifier.predict_proba(sensors[risk_features])[:, 1]
    test_score = classifier.predict_proba(test_x)[:, 1]
    roc_auc = float(roc_auc_score(test_y, test_score)) if test_y.nunique() > 1 else None
    joblib.dump(classifier, MODELS_DIR / 'pump_risk_logreg.joblib')
else:
    sensors['failure_risk_score'] = sensors['anomaly_flag'].astype(float)
    roc_auc = None

sensors.to_sql('dm_pump_risk', engine, if_exists='replace', index=False)
metrics = {'roc_auc': roc_auc, 'contamination': 0.05, 'model': 'LogisticRegression_balanced'}
(MODELS_DIR / 'pump_risk_metrics.json').write_text(json.dumps(metrics, indent=2), encoding='utf-8')
metrics

{'roc_auc': 1.0, 'contamination': 0.05, 'model': 'LogisticRegression_balanced'}

## 5. Trino + S3 parquet demo

In [6]:
import time

trino_engine = create_engine('trino://trino@trino:8080/hive')
last_error = None
for _ in range(24):
    try:
        pd.read_sql('SELECT 1', trino_engine)
        break
    except Exception as exc:
        last_error = exc
        time.sleep(5)
else:
    raise RuntimeError('Trino endpoint did not become ready') from last_error

statements = [
    'CREATE SCHEMA IF NOT EXISTS hive.oil',
    'DROP TABLE IF EXISTS hive.oil.production_parquet',
    '''
    CREATE TABLE hive.oil.production_parquet (
        prod_id integer,
        well_id integer,
        date date,
        oil_ton double,
        gas_m3 double,
        water_m3 double,
        energy_kwh double,
        downtime_hours double,
        temperature double,
        pressure double,
        dt date
    )
    WITH (
        external_location = 's3://oil-lake/bronze/production',
        format = 'PARQUET',
        partitioned_by = ARRAY['dt']
    )
    ''',
    "CALL hive.system.sync_partition_metadata('oil', 'production_parquet', 'FULL')",
]
with trino_engine.begin() as connection:
    for statement in statements:
        connection.execute(text(statement))

pd.read_sql('''
SELECT dt, count(*) AS rows_count, round(sum(oil_ton), 2) AS oil_ton
FROM hive.oil.production_parquet
GROUP BY dt
ORDER BY dt
LIMIT 10
''', trino_engine)

,dt,rows_count,oil_ton
0,2025-10-01,10,1435.0
1,2025-10-02,10,1434.6
2,2025-10-03,10,1438.4
3,2025-10-04,10,1445.8
4,2025-10-05,10,1442.0
5,2025-10-06,10,1437.2
6,2025-10-07,10,1428.2
7,2025-10-08,10,1432.8
8,2025-10-09,10,1443.4
9,2025-10-10,10,1437.0
